# 02 — DenseNet121 detector (tiny smoke test)

Uses `samples_tiny/manifest.csv`. The model is not validated for detecting AI edits; this notebook only verifies training, checkpointing, and predictions.

In [ ]:
%pip install -q timm torch torchvision Pillow pytest

In [ ]:
from pathlib import Path
import os
GITHUB_REPO = "YOUR_USERNAME/YOUR_REPO"  # Change this
assert GITHUB_REPO != "YOUR_USERNAME/YOUR_REPO"
repo_dir = Path("/content") / GITHUB_REPO.split("/")[-1]
if not repo_dir.exists():
    !git clone https://github.com/{GITHUB_REPO}.git {repo_dir}
assert (repo_dir / "samples_tiny/manifest.csv").is_file(), "Check samples_tiny folder and clone path"
import sys
sys.path.insert(0, str(repo_dir / "src"))

In [ ]:
from imgcompliance.detector import train, Detector
manifest = repo_dir / "samples_tiny" / "manifest.csv"
checkpoint = Path("/content/detector_head.pt")
detector = train(manifest, epochs=3, batch_size=6, output_path=checkpoint)
print("Checkpoint saved:", checkpoint)

In [ ]:
import csv
with manifest.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
for row in rows:
    path = manifest.parent / row["filename"]
    print(path.name, "ground_truth=", row["label"], "prediction=", detector.predict(path))

In [ ]:
# Check checkpoint reload
reloaded = Detector(checkpoint)
print("Reloaded model:", reloaded.predict(manifest.parent / rows[0]["filename"]))